# Training Pipelines, Distributed Training and GPU Cost Control

Notebook for **Chapter 6** of the *mlops* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/mlops/ch06.html

Tip: change a number, predict the output, then run the cell again.

## 6.1 Training Pipelines and Orchestration

In [ ]:
from graphlib import TopologicalSorter
dag = {"extract": set(), "validate": {"extract"}, "features": {"validate"}, "train": {"features"},
       "evaluate": {"train"}, "register": {"evaluate"}, "report": {"evaluate"}}
print("run order:", list(TopologicalSorter(dag).static_order()))
bad = dict(dag); bad["extract"] = {"register"}
try: list(TopologicalSorter(bad).static_order())
except Exception as e: print("cycle error:", type(e).__name__, e.args[1])

In [ ]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta
with DAG("churn_retrain", start_date=datetime(2026, 1, 1), schedule="@weekly", catchup=False,
         default_args={"retries": 2, "retry_delay": timedelta(minutes=5)}) as dag:
    extract = PythonOperator(task_id="extract", python_callable=run_extract)
    validate = PythonOperator(task_id="validate", python_callable=run_validate)
    train = PythonOperator(task_id="train", python_callable=run_train)
    extract >> validate >> train

In [ ]:
import json, pathlib
OUT = pathlib.Path("/tmp/lab/idem"); OUT.mkdir(exist_ok=True)
def append_job(day, rows):             # NOT idempotent
    with open(OUT/"append.jsonl", "a") as f:
        for r in rows: f.write(json.dumps({"day": day, "v": r}) + "\n")
def overwrite_job(day, rows):          # idempotent: one file per day, replaced whole
    (OUT/f"day={day}.json").write_text(json.dumps(rows))
for _ in range(2): append_job("2026-03-01", [1, 2, 3]); overwrite_job("2026-03-01", [1, 2, 3])
print("append  rows after 2 runs:", len((OUT/"append.jsonl").read_text().splitlines()))
print("overwrite rows after 2 runs:", len(json.loads((OUT/"day=2026-03-01.json").read_text())))

## 6.2 Distributed Training Basics

In [ ]:
import numpy as np
rng = np.random.default_rng(0); X = rng.normal(size=(8, 3)); y = rng.normal(size=8); w = np.zeros(3)
grad = lambda Xs, ys, w: 2 * Xs.T @ (Xs @ w - ys) / len(ys)       # gradient of mean squared error
full = grad(X, y, w)
shards = [grad(X[i*2:(i+1)*2], y[i*2:(i+1)*2], w) for i in range(4)]    # 4 workers, 2 rows each
print("full-batch grad:", np.round(full, 4)); print("mean of 4 worker grads:", np.round(np.mean(shards, 0), 4))
print("equal:", np.allclose(full, np.mean(shards, 0)))

In [ ]:
compute, comm = 100.0, 8.0
for N in [1,2,4,8,16]:
    t = compute/N + (comm if N>1 else 0)
    print(f"GPUs={N:>2}  step={t:6.2f} ms  speedup={compute/t:5.2f}  efficiency={compute/t/N*100:5.1f}%")

GPUs= 1  step=100.00 ms  speedup= 1.00  efficiency=100.0%
GPUs= 2  step= 58.00 ms  speedup= 1.72  efficiency= 86.2%
GPUs= 4  step= 33.00 ms  speedup= 3.03  efficiency= 75.8%
GPUs= 8  step= 20.50 ms  speedup= 4.88  efficiency= 61.0%
GPUs=16  step= 14.25 ms  speedup= 7.02  efficiency= 43.9%


In [ ]:
# train.py (sketch, not run here)
import torch, torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
dist.init_process_group("nccl")
rank = dist.get_rank(); torch.cuda.set_device(rank)
model = DDP(MyModel().cuda(rank), device_ids=[rank])
# use DistributedSampler so each rank sees a different shard; call sampler.set_epoch(epoch) each epoch

## 6.3 GPU Management and Cost Control

In [ ]:
import numpy as np
print("float16 max:", np.finfo(np.float16).max, "float32 max:", np.finfo(np.float32).max)
with np.errstate(over="ignore"):
    print("60000 + 60000 in float16:", np.float16(60000) + np.float16(60000))
print("1.0 + 0.0001 in float16:", np.float16(1.0) + np.float16(0.0001), "| in float32:", np.float32(1.0)+np.float32(0.0001))

float16 max: 6.55e+04 float32 max: 3.4028235e+38
60000 + 60000 in float16: inf
1.0 + 0.0001 in float16: 1.0 | in float32: 1.0001


In [ ]:
import random
od, spot, need_h = 3.00, 1.00, 20.0
def sim(ckpt_every_h, preempt_per_h=0.2, seed=1, restart_cost_h=0.1):
    rng = random.Random(seed); done = 0.0; billed = 0.0; saved = 0.0; since = 0.0; step = 0.05
    while saved < need_h:
        billed += step; done += step; since += step
        if rng.random() < preempt_per_h * step:      # preempted: lose work since last checkpoint
            done = saved; since = 0.0; billed += restart_cost_h
        elif since >= ckpt_every_h: saved = done; since = 0.0
        if done >= need_h: saved = need_h
    return billed
print(f"on-demand: {need_h:.0f} h x ${od} = ${need_h*od:.2f}")
for c in [0.25, 1, 4, 20]:
    h = sim(c); print(f"spot, checkpoint every {c:>5} h: billed {h:5.1f} GPU-h -> ${h*spot:6.2f}")

## 6.4 Distributed Training Arithmetic: Memory, Communication and Failures

Section 6.2 showed data parallelism and its speedup limits. Large models break that picture in three ways: the model state does not fit on one GPU, the gradient exchange takes longer than the compute, and with hundreds of GPUs something fails every few hours. This section gives you the arithmetic for each. All numbers are computed by code that ran here. I did not train anything on a real cluster: PyTorch and GPUs are not available here, so the training snippet is labelled not run here.

In [ ]:
# Memory per parameter in mixed-precision training with Adam
fp16_weights, fp16_grads = 2, 2
fp32_master, adam_m, adam_v = 4, 4, 4
per_param = fp16_weights + fp16_grads + fp32_master + adam_m + adam_v
print("bytes per parameter:", per_param)

params = 7e9
print("7B model state:", round(params * per_param / 1e9), "GB (before activations)")

def per_gpu(stage, n):
    """GB of model state per GPU. stage 0 = plain data parallel."""
    opt = 12 / (n if stage >= 1 else 1)        # fp32 master + m + v
    grad = 2 / (n if stage >= 2 else 1)
    weight = 2 / (n if stage >= 3 else 1)
    return params * (opt + grad + weight) / 1e9

names = {0: "DDP (copy everything)", 1: "ZeRO-1 (shard optimizer)", 2: "ZeRO-2 (+ gradients)", 3: "ZeRO-3 / FSDP (+ weights)"}
for stage in range(4):
    print(f"{names[stage]:28s} 8 GPUs: {per_gpu(stage, 8):6.1f} GB   64 GPUs: {per_gpu(stage, 64):6.1f} GB")

bytes per parameter: 16
7B model state: 112 GB (before activations)
DDP (copy everything)        8 GPUs:  112.0 GB   64 GPUs:  112.0 GB
ZeRO-1 (shard optimizer)     8 GPUs:   38.5 GB   64 GPUs:   29.3 GB
ZeRO-2 (+ gradients)         8 GPUs:   26.2 GB   64 GPUs:   15.5 GB
ZeRO-3 / FSDP (+ weights)    8 GPUs:   14.0 GB   64 GPUs:    1.8 GB


In [ ]:
# Ring all-reduce time: each GPU sends and receives 2 (N-1)/N x bytes.
def allreduce_s(nbytes, n, bandwidth):
    return 2 * (n - 1) / n * nbytes / bandwidth
grads = 7e9 * 2                          # fp16 gradients of a 7B model: 14 GB
for name, bw in (("NVLink 450 GB/s", 450e9), ("InfiniBand 50 GB/s", 50e9), ("Ethernet 100 Gb/s = 12.5 GB/s", 12.5e9)):
    print(f"{name:32s} {allreduce_s(grads, 8, bw):6.2f} s per step")

NVLink 450 GB/s                    0.05 s per step
InfiniBand 50 GB/s                 0.49 s per step
Ethernet 100 Gb/s = 12.5 GB/s      1.96 s per step


In [ ]:
import numpy as np
# Gradient accumulation gives the same gradient as one big batch.
rng = np.random.default_rng(0)
X = rng.normal(size=(64, 3)); y = X @ np.array([1.0, -2.0, 0.5]) + rng.normal(scale=0.1, size=64)
w = np.zeros(3)

def grad(Xb, yb, w):                   # gradient of mean squared error
    return 2 * Xb.T @ (Xb @ w - yb) / len(yb)

full = grad(X, y, w)
accum = np.zeros(3)
for i in range(4):                     # 4 micro-batches of 16
    accum += grad(X[i*16:(i+1)*16], y[i*16:(i+1)*16], w) / 4
print(np.round(full, 6))
print(np.round(accum, 6))
print("same:", np.allclose(full, accum))
# effective batch = micro_batch x accumulation_steps x GPUs
print("effective batch:", 16 * 4 * 8)

[-2.265977  2.899352 -0.855204]
[-2.265977  2.899352 -0.855204]
same: True
effective batch: 512


In [ ]:
# Pipeline parallel: with p stages and m micro-batches, the idle "bubble" fraction is (p-1)/(m+p-1)
p = 4
for m in (1, 4, 8, 16, 32, 64):
    print(f"p={p}, micro-batches={m:3d}: bubble {(p-1)/(m+p-1):.1%}")

p=4, micro-batches=  1: bubble 75.0%
p=4, micro-batches=  4: bubble 42.9%
p=4, micro-batches=  8: bubble 27.3%
p=4, micro-batches= 16: bubble 15.8%
p=4, micro-batches= 32: bubble 8.6%
p=4, micro-batches= 64: bubble 4.5%


In [ ]:
import math, random
# Checkpoint interval. C = minutes to write a checkpoint, M = mean minutes between failures.
C_min, M = 3.0, 360.0
print("Young-Daly interval:", round(math.sqrt(2 * C_min * M), 1), "minutes")

def waste_formula(T):                      # first-order estimate of the lost fraction
    return C_min / T + T / (2 * M)

def simulate(T, horizon=3_000_000, seed=1):
    rnd = random.Random(seed)
    t = 0.0; saved = 0.0                   # saved = useful minutes safely written to a checkpoint
    fail_at = rnd.expovariate(1 / M)
    while t < horizon:
        if t + T + C_min <= fail_at:       # a whole work-then-checkpoint cycle completes
            t += T + C_min; saved += T
        else:                              # a failure inside the cycle loses the unsaved work
            t = fail_at + 5.0              # 5 minutes to restart
            fail_at = t + rnd.expovariate(1 / M)
    return 1 - saved / t

print("interval  formula  simulated waste")
for T in (10, 20, 46, 90, 180, 360):
    print(f"{T:6d} min  {waste_formula(T):6.1%}   {simulate(T):6.1%}")

Young-Daly interval: 46.5 minutes
interval  formula  simulated waste
    10 min   31.4%    25.5%
    20 min   17.8%    17.0%
    46 min   12.9%    13.5%
    90 min   15.8%    16.4%
   180 min   26.7%    25.7%
   360 min   50.8%    43.7%


In [ ]:
import torch
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import MixedPrecision

torch.distributed.init_process_group("nccl")
model = build_model().cuda()
model = FSDP(
    model,
    mixed_precision=MixedPrecision(param_dtype=torch.bfloat16,
                                   reduce_dtype=torch.bfloat16),
)
opt = torch.optim.AdamW(model.parameters(), lr=2e-4)
for step, (x, y) in enumerate(loader):
    loss = loss_fn(model(x.cuda()), y.cuda())
    loss.backward()                 # gradients are reduce-scattered here
    if (step + 1) % 4 == 0:         # gradient accumulation of 4
        opt.step(); opt.zero_grad()

## Chapter recap

## Practice set

Problem 6.1 Order these tasks: train, extract, evaluate, features. **Answer:** extract, features, train, evaluate.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/mlops/ch06.html